In [ ]:
!pip install transformers datasets torchaudio librosa

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 65.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 21.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 36.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 994.4 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 63.5 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalling nvidia-nvjitlink-cu12-12.5.82:
      Successfully uninstalled nvidia-nvjitli

whisper 모델, 프로세서 로드

In [ ]:
from transformers import WhisperProcessor, WhisperForConditionalGeneration

# 모델과 프로세서 로드
model_name = "openai/whisper-tiny"
processor = WhisperProcessor.from_pretrained(model_name)
model = WhisperForConditionalGeneration.from_pretrained(model_name)

/usr/local/lib/python3.11/dist-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/usr/local/lib/python3.11/dist-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/usr/local/lib/python3.11/dist-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/usr/local/lib/python3.11/dist-packages/huggingface_hub/file_download.py:943: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `for

preprocessor_config.json:   0%|          | 0.00/185k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/283k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/836k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.48M [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/2.19k [00:00<?, ?B/s]

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


config.json:   0%|          | 0.00/1.98k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/151M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/3.75k [00:00<?, ?B/s]

In [ ]:
import torchaudio

음성 인식(ASR) 모델 학습을 위한 데이터 전처리 함수

In [ ]:
def prepare_dataset(batch):
    input_features = []
    labels = []
    for audio_path, text in zip(batch["audio"], batch["text"]):
        try:
            speech_array, sr = torchaudio.load(audio_path)
            if sr != 16000:
                speech_array = torchaudio.transforms.Resample(sr, 16000)(speech_array)
            inputs = processor(speech_array[0], sampling_rate=16000, return_tensors="pt")
            input_features.append(inputs.input_features.squeeze(0))
            labels.append(processor.tokenizer(text).input_ids)
        except:
            input_features.append(None)
            labels.append(None)
    return {"input_features": input_features, "labels": labels}


In [ ]:
valid_dataset = dataset

음성 데이터셋을 단위로 분할하여 전처리하고 저장

In [ ]:
import math
import gc

# 나누고 싶은 단위 설정 (예: 약 8000개씩)
chunk_size = 5000
num_chunks = math.ceil(len(valid_dataset) / chunk_size)

# 전처리 함수
def process_and_save(split, idx):
    processed = split.map(
        prepare_dataset,
        batched=True,
        batch_size=8,
        remove_columns=["audio", "text"],
        num_proc=1,
        desc=f"Whisper 전처리 중 - Part {idx}"
    )
    processed.save_to_disk(f"/content/drive/MyDrive/whisper_cached_part_{idx}")
    del processed
    gc.collect()

# 자동 분할 및 전처리 실행
for idx in range(num_chunks):
    start = idx * chunk_size
    end = min(start + chunk_size, len(valid_dataset))
    split = valid_dataset.select(range(start, end))
    process_and_save(split, idx)

Whisper 전처리 중 - Part 0:   0%|          | 0/5000 [00:00<?, ? examples/s]

Saving the dataset (0/10 shards):   0%|          | 0/5000 [00:00<?, ? examples/s]

Whisper 전처리 중 - Part 1:   0%|          | 0/5000 [00:00<?, ? examples/s]

Saving the dataset (0/10 shards):   0%|          | 0/5000 [00:00<?, ? examples/s]

Whisper 전처리 중 - Part 2:   0%|          | 0/5000 [00:00<?, ? examples/s]

...

Saving the dataset (0/10 shards):   0%|          | 0/5000 [00:00<?, ? examples/s]

Whisper 전처리 중 - Part 18:   0%|          | 0/5000 [00:00<?, ? examples/s]

Saving the dataset (0/10 shards):   0%|          | 0/5000 [00:00<?, ? examples/s]

Whisper 전처리 중 - Part 19:   0%|          | 0/5000 [00:00<?, ? examples/s]

Saving the dataset (0/10 shards):   0%|          | 0/5000 [00:00<?, ? examples/s]

분할 저장된 데이터셋 파트들을 하나로 병합하는 작업

In [ ]:
from datasets import load_from_disk, concatenate_datasets
import math

# 병합할 파트 수 (셀 1에서 사용한 num_chunks 값과 같아야 함)
# num_chunks = math.ceil(len(valid_dataset) / 8000)

# 파트 개수 직접 지정 (폴더가 0~19까지 총 20개이므로)
num_chunks = 20

# 저장된 파트 불러오기 및 병합
paths = [f"/content/drive/MyDrive/whisper_cached_part_{i}" for i in range(num_chunks)]
datasets = [load_from_disk(p) for p in paths]
merged_dataset = concatenate_datasets(datasets)

# 최종 저장
merged_dataset.save_to_disk("/content/drive/MyDrive/whisper_combined")

/usr/local/lib/python3.11/dist-packages/datasets/table.py:1421: FutureWarning: promote has been superseded by promote_options='default'.
  table = cls._concat_blocks(blocks, axis=0)


OSError: [Errno 28] No space left on device: '/content/drive/MyDrive/whisper_cached_part_16/data-00005-of-00010.arrow' -> '/tmp/tmpf5a4e3ei/content/drive/MyDrive/whisper_cached_part_16/data-00005-of-00010.arrow'

In [ ]:
!pip install transformers datasets accelerate evaluate jiwer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.0/84.0 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 54.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 44.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 28.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 24.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 752.3 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 20.4 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
 

In [ ]:
!pip list | grep -E 'transformers|datasets|torch'
# transformers==4.35.2, datasets==2.18.0, torch==2.0.1 확인

병합한 데이터셋 로드

In [ ]:
from transformers import WhisperForConditionalGeneration, WhisperProcessor
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer
from datasets import load_from_disk
import evaluate
from transformers import Trainer

# 1. 데이터셋 로드
merged_dataset = load_from_disk("/content/drive/MyDrive/whisper_combined")

# 2. 데이터셋 분할 (80 train, 20 validation)
dataset = merged_dataset.train_test_split(test_size=0.2, seed=42)

메트릭 계산 함수 정의 (WER: 단어 오류율)

In [ ]:
wer_metric = evaluate.load("wer")

def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids
    label_str = processor.batch_decode(label_ids, skip_special_tokens=True)
    pred_str = processor.batch_decode(pred_ids, skip_special_tokens=True)
    wer = wer_metric.compute(predictions=pred_str, references=label_str)
    return {"wer": wer}

whisper 훈련 설정하기

In [ ]:
training_args = Seq2SeqTrainingArguments(
    output_dir="./whisper-results",  # 결과 저장 경로
    per_device_train_batch_size=16,   # 배치 크기
    gradient_accumulation_steps=1,
    learning_rate=1e-5,
    num_train_epochs=3,
    fp16=True,  # GPU 사용 시
    eval_strategy="steps",
    save_steps=500,
    eval_steps=500,
    logging_dir="./logs",
)

trainer 초기화 시키기

In [ ]:
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],  # 분할된 train
    eval_dataset=dataset["test"],    # 분할된 validation (test로 자동 명명됨)
    tokenizer=processor.tokenizer,   # 텍스트 토크나이저
    compute_metrics=compute_metrics  # WER 계산 함수 추가
)

훈련

In [ ]:
trainer.train()